## Cálculo de similitudes y asignación de claves de la Muestra 2

Una vez concluido el proceso de preprocesamiento y homologación de la Muestra 2, se desarrolló un procedimiento para identificar las correspondencias entre los vehículos de la muestra y los registros del catálogo vehicular.

Al igual que en la Muestra 1, se utilizaron **el coeficiente de Gower y una combinación de las medidas de similitud textual Token Set Ratio y Token Sort Ratio**, con el propósito de evaluar tanto las características técnicas de los vehículos como las semejanzas entre sus descripciones.

Estas medidas se integraron mediante una similitud global ponderada, utilizada para seleccionar el registro del catálogo con mayor semejanza respecto a cada vehículo de la muestra. Posteriormente, se incorporó una segunda fase de asignación destinada a los vehículos que no obtuvieron una clave mediante el procedimiento inicial.

### 1. Preparación de las variables para la comparación

Inicialmente, se recuperaron la Muestra 2 y el catálogo vehicular previamente homologados, conservando las transformaciones realizadas durante la etapa de ingeniería de datos.

Posteriormente, se construyó una descripción adicional para cada registro del catálogo a partir de la concatenación de las siguientes variables:

- `MARCA`: fabricante del vehículo.
- `TIPO`: denominación o línea del vehículo.
- `VERSION`: versión o conjunto de características particulares.
- `TRANSMISIÓN`: tipo de transmisión del vehículo.

Para incorporar la transmisión en estas descripciones, los códigos `A` y `S` se convirtieron a `AUT` y `STD`, respectivamente. Las categorías distintas de estas equivalencias, como `A/S`, no recibieron una terminación específica.

Durante la construcción de las descripciones, los valores faltantes se sustituyeron por cadenas vacías únicamente en las columnas auxiliares utilizadas para la concatenación. Posteriormente, se normalizaron los espacios entre palabras, sin modificar las variables originales del catálogo.

### 2. Similitud de Gower

El coeficiente de Gower permite comparar observaciones a partir de múltiples variables, incluso cuando estas presentan distintas escalas de medición. Sin embargo, para este proyecto se optó por tratar las seis variables seleccionadas como **categóricas nominales**, incluyendo el año del modelo y la capacidad de ocupantes del vehículo.

Las variables consideradas fueron:

| Variable de la muestra | Variable del catálogo |
|---|---|
| modelo | MODELO |
| Marca | MARCA |
| transmision | TRANSMISIÓN |
| motor | MOTOR |
| AC | AIRE ACONDICIONADO |
| Ocupantes | OCUPANTES |

A diferencia de la Muestra 1, se incorporó la variable `Ocupantes`, que representa la capacidad de pasajeros del vehículo y que había sido previamente homologada a un formato numérico entero.

Aunque `modelo` y `Ocupantes` contienen valores numéricos, ambas variables se trataron como categorías. Por lo tanto, únicamente se evaluó su coincidencia exacta, sin establecer grados de semejanza entre años o capacidades diferentes.

Para cada variable se asignó una similitud de 1 cuando los valores coincidían y de 0 cuando eran diferentes. Posteriormente, se calculó el promedio de las coincidencias observadas:

$$
S_G(i,j)=
\frac{\displaystyle\sum_{k=1}^{p}\delta_{ijk}s_{ijk}}
{\displaystyle\sum_{k=1}^{p}\delta_{ijk}}
$$

Donde:

- $S_G(i,j)$ representa la similitud de Gower entre los vehículos $i$ y $j$.
- $p=6$ es el número de variables categóricas consideradas.
- $s_{ijk}=1$ si ambos vehículos coinciden en la variable $k$, y $0$ en caso contrario.
- $\delta_{ijk}=1$ cuando ambos registros tienen información disponible para la variable $k$, y $0$ cuando alguno presenta un valor faltante.

Las seis variables recibieron la misma importancia dentro del cálculo.

Para facilitar las comparaciones, las categorías de ambas bases de datos se codificaron conjuntamente mediante `pd.factorize()`, identificando los valores faltantes con el código `-1`. Cuando alguno de los vehículos presentaba información faltante en determinada variable, esta se excluía del cálculo, ajustando el denominador al número de variables comparables. Si ninguna variable estaba disponible, la similitud se consideraba indefinida (`NaN`).

Se seleccionó esta medida porque permite evaluar conjuntamente características relevantes para la identificación del vehículo, como el año del modelo, la marca, la transmisión, la configuración del motor, el aire acondicionado y la capacidad de ocupantes.

El resultado se encuentra entre 0 y 1, donde **1 representa coincidencia en todas las variables comparables y 0 representa ausencia de coincidencias**.

### 3. Similitud textual

Para comparar las variables textuales `Tipo` y `descripcion`, se utilizaron dos medidas de similitud de cadenas de caracteres, implementadas mediante la biblioteca `RapidFuzz` de Python.

**Token Set Ratio**

Esta medida compara las cadenas a partir de las palabras que las componen, identificando los términos compartidos y aquellos que aparecen exclusivamente en cada texto.

Su principal ventaja consiste en reconocer semejanzas elevadas entre cadenas que contienen palabras en diferente orden o cantidades distintas de información.

No obstante, presenta una limitación importante: cuando una cadena contiene todas las palabras de otra, pero incorpora términos adicionales, Token Set Ratio puede asignar una similitud de 100 %, aun cuando las descripciones no sean completamente equivalentes.

**Token Sort Ratio**

Esta medida separa las cadenas en palabras, las ordena alfabéticamente y posteriormente compara los textos resultantes.

A diferencia de Token Set Ratio, mantiene una mayor sensibilidad ante la presencia de palabras adicionales o ausentes. Por lo tanto, permite reducir las puntuaciones de similitud que podrían resultar excesivamente elevadas al utilizar únicamente Token Set Ratio.

**Combinación de las similitudes textuales**

Considerando las características de ambas medidas, se decidió construir una similitud textual ponderada que aprovechara la capacidad de Token Set Ratio para identificar cadenas semejantes y, simultáneamente, incorporara Token Sort Ratio para penalizar las diferencias derivadas de palabras adicionales.

Ambas medidas producen puntuaciones entre 0 y 100, que fueron divididas entre 100 para expresarlas en el intervalo de 0 a 1.

La similitud textual se definió como:

$$
S_T(i,j)=0.75S_{\text{set}}(i,j)+0.25S_{\text{sort}}(i,j)
$$

Donde:

- $S_T(i,j)$ es la similitud textual combinada.
- $S_{\text{set}}(i,j)$ es la similitud obtenida mediante Token Set Ratio.
- $S_{\text{sort}}(i,j)$ es la similitud obtenida mediante Token Sort Ratio.

Se asignó una ponderación de **75 % a Token Set Ratio y 25 % a Token Sort Ratio**, otorgando mayor importancia a la identificación de palabras compartidas, pero reduciendo la posibilidad de asignar una similitud perfecta a descripciones con información adicional.

Esta combinación se utilizó tanto para comparar las descripciones completas de los vehículos como para evaluar las denominaciones de la variable `Tipo`.

Cuando alguna de las cadenas comparadas presentaba un valor faltante (`NaN`) o estaba vacía, la similitud textual se consideraba indefinida, en lugar de asignarle una puntuación de cero.

### 4. Construcción de la similitud global ponderada

Una vez definidas las medidas anteriores, se estableció una similitud global integrada por tres componentes:

| Componente | Ponderación |
|---|---:|
| Similitud de Gower | 50 % |
| Similitud textual de descripción | 40 % |
| Similitud textual de Tipo | 10 % |
| **Total** | **100 %** |

Cuando los tres componentes estaban disponibles, la similitud global se calculó mediante:

$$
S_{\text{global}}(i,j)=0.50S_G(i,j)+0.40S_D(i,j)+0.10S_{\text{Tipo}}(i,j)
$$

Donde:

- $S_G(i,j)$ corresponde a la similitud de Gower.
- $S_D(i,j)$ corresponde a la similitud textual entre las descripciones completas.
- $S_{\text{Tipo}}(i,j)$ corresponde a la similitud textual entre las denominaciones de `Tipo`.

Las ponderaciones se establecieron considerando la importancia asignada a cada componente dentro del procedimiento de identificación vehicular.

Se otorgó el mayor peso a **Gower (50 %)**, debido a que incorpora características técnicas y de equipamiento consideradas relevantes para distinguir vehículos, incluyendo el año del modelo, la transmisión, el motor, el aire acondicionado y la capacidad de ocupantes.

Por otra parte, se asignó un **40 % a la similitud de descripción**, ya que permite identificar semejanzas entre las denominaciones completas de los vehículos, considerando tanto los términos compartidos como las diferencias producidas por información adicional.

Finalmente, se otorgó un **10 % a la similitud de Tipo**, utilizándola como un componente complementario para evaluar la correspondencia entre las denominaciones de las líneas o tipos de vehículos.

**Distinción entre la similitud de descripción y la similitud de Tipo**

Aunque ambas comparaciones utilizan la misma medida textual, evalúan la información con distinto nivel de especificidad.

La similitud de descripción compara el texto completo del vehículo de la muestra con la descripción generada del catálogo, integrada por marca, tipo, versión y transmisión. En cambio, la similitud de `Tipo` compara exclusivamente las denominaciones de esta variable.

Esta distinción es relevante porque dos registros pueden presentar descripciones considerablemente semejantes, pero diferir en la denominación de `Tipo`.

Debe señalarse que ambas comparaciones no son independientes, ya que `TIPO` forma parte de la descripción generada del catálogo. En consecuencia, su evaluación adicional otorga deliberadamente una mayor importancia a esta característica dentro de la puntuación global.

**Tratamiento de componentes faltantes**

Cuando alguno de los tres componentes presentaba una similitud indefinida (`NaN`), el algoritmo calculaba el promedio ponderado utilizando únicamente los componentes disponibles.

Para ello, se excluía el peso correspondiente al componente faltante y se ajustaba el denominador mediante la siguiente expresión:

$$
S_{\text{global}}(i,j)=
\frac{
\displaystyle\sum_{k=1}^{3}\delta_{ijk}w_kS_k(i,j)
}{
\displaystyle\sum_{k=1}^{3}\delta_{ijk}w_k
}
$$

Donde:

- $w_k$ representa la ponderación del componente $k$.
- $S_k(i,j)$ representa la similitud obtenida en dicho componente.
- $\delta_{ijk}=1$ si la similitud está disponible y $0$ si es indefinida.

Por ejemplo, si la similitud de `Tipo` no podía calcularse, pero Gower y descripción sí estaban disponibles, la similitud global se obtenía mediante:

$$
S_{\text{global}}(i,j)=
\frac{0.50S_G(i,j)+0.40S_D(i,j)}{0.90}
$$

De esta manera, los componentes disponibles conservaban sus ponderaciones relativas, sin sustituir las similitudes faltantes por cero. Si ninguno estaba disponible, la similitud global permanecía como `NaN`.

### 5. Filtrado textual y preselección de candidatos

Para reducir el número de comparaciones necesarias y concentrar el análisis en los vehículos con mayor semejanza descriptiva, se implementó un procedimiento de **preselección de candidatos basado en la similitud textual de las descripciones**.

Este filtrado utilizó la combinación de Token Set Ratio y Token Sort Ratio definida anteriormente, permitiendo identificar las descripciones del catálogo que compartían una mayor semejanza con la del vehículo analizado.

La finalidad de esta etapa fue seleccionar los registros descriptivamente más próximos antes de incorporar las características categóricas mediante Gower y la comparación independiente de `Tipo`.

Se estableció como criterio de preselección conservar **las 15 descripciones distintas con mayor similitud textual**. Esta cantidad representa el número máximo de descripciones seleccionadas, no necesariamente el número de registros individuales, debido a que una misma descripción puede estar asociada a varios registros del catálogo.

Es importante destacar que **el filtrado textual no constituye el criterio definitivo de asignación de claves**, sino una etapa preliminar que permite delimitar el conjunto de candidatos sobre el cual se calculará posteriormente la similitud global ponderada.

### 6. Procedimiento inicial de asignación de claves

Una vez definidas las medidas de similitud y el criterio de preselección, se implementó un algoritmo para identificar la correspondencia más semejante entre cada vehículo de la Muestra 2 y los registros del catálogo vehicular.

El procedimiento se desarrolló en el siguiente orden:

1. **Filtrado por marca:** inicialmente, se seleccionaron los registros del catálogo cuya marca coincidía exactamente con la del vehículo de la muestra. Si no existían registros correspondientes, el vehículo permanecía sin clave asignada.

2. **Comparación de descripciones:** dentro del conjunto de vehículos de la misma marca, se identificaron las descripciones distintas y se calculó su similitud textual respecto a la descripción del vehículo analizado, utilizando la combinación ponderada de Token Set Ratio y Token Sort Ratio.

3. **Preselección de candidatos:** se seleccionaron hasta 15 descripciones distintas con mayor similitud textual y se recuperaron todos los registros del catálogo asociados a ellas. La selección priorizó las similitudes válidas, aunque podía incluir descripciones con puntuaciones indefinidas cuando existían menos de 15 descripciones evaluables.

4. **Cálculo de Gower:** para cada registro candidato, se calculó la similitud de Gower utilizando las seis variables categóricas previamente definidas: `modelo`, `Marca`, `transmision`, `motor`, `AC` y `Ocupantes`. Las variables con valores faltantes se excluyeron de cada comparación.

5. **Eliminación de duplicidades:** cuando varios registros del catálogo compartían una misma descripción generada, se conservó únicamente aquel con mayor similitud de Gower.

6. **Comparación de Tipo:** para los candidatos restantes, se calculó la similitud textual entre las denominaciones de `Tipo` de la muestra y del catálogo, utilizando nuevamente la combinación de Token Set Ratio y Token Sort Ratio.

7. **Cálculo de la similitud global:** se integraron las tres puntuaciones mediante la combinación ponderada previamente establecida, ajustando el denominador cuando alguno de los componentes era indefinido.

8. **Selección del mejor candidato:** se ordenaron los candidatos de mayor a menor similitud global y se seleccionó aquel con la puntuación más alta.

9. **Asignación de clave:** finalmente, se estableció un umbral de aceptación de **0.80**. Si la similitud global del mejor candidato era estrictamente superior a dicho valor, se asignaba al vehículo de la muestra el identificador `ID` correspondiente del catálogo. En caso contrario, el registro permanecía sin clave asignada.

Este procedimiento se aplicó a todas las observaciones de la Muestra 2, almacenando los identificadores obtenidos en la columna `ID_asignado`.

### 7. Segunda fase: asignación complementaria de claves

Con el propósito de incrementar la cobertura del claveteo, se incorporó una segunda fase dirigida exclusivamente a los vehículos que permanecieron sin clave asignada después del procedimiento inicial.

Esta etapa empleó un criterio de correspondencia más flexible, basado en las variables `Marca` y `Tipo`, así como en el valor comercial de los vehículos registrados en el catálogo.

A diferencia de la primera fase, **no se calculó nuevamente la similitud global ponderada ni se establecieron restricciones relacionadas con el año-modelo, la versión o las características de equipamiento**.

El procedimiento se desarrolló de la siguiente manera:

1. **Normalización de denominaciones:** se estandarizaron las cadenas de texto de `Marca` y `Tipo`, eliminando los espacios exteriores y convirtiendo las denominaciones a mayúsculas para facilitar su comparación.

2. **Filtrado por marca y tipo:** para cada vehículo sin clave, se identificaron los registros del catálogo que presentaban coincidencias en ambas variables. No se exigió correspondencia en las características técnicas restantes.

3. **Comparación de valores comerciales:** cuando existían varios registros del catálogo con la misma combinación de marca y tipo, se evaluaron sus valores comerciales. Para ello, se utilizó la variable `COMERCIAL`, convirtiendo sus valores a formato numérico y excluyendo aquellos sin información válida.

4. **Selección del registro:** entre los candidatos con correspondencia de marca y tipo, se seleccionó aquel que presentaba el **mayor valor comercial** dentro del catálogo vehicular.

5. **Asignación de clave:** se recuperó el identificador `ID` del registro seleccionado y se incorporó a la muestra. Cuando no existía una correspondencia de marca y tipo con valor comercial disponible, el vehículo permanecía sin clave asignada.

Esta segunda fase permitió establecer correspondencias para vehículos que no habían superado el umbral de similitud global inicial, utilizando una regla de asignación basada en las denominaciones comerciales y en el mayor valor comercial disponible.

La elección del vehículo con mayor valor comercial constituyó un criterio de selección entre registros que compartían marca y tipo, sin que ello implicara una coincidencia exacta en el año-modelo, la versión o el equipamiento.

Las claves obtenidas durante la primera fase se conservaron sin modificaciones, aplicando este procedimiento exclusivamente a los registros pendientes de asignación.

### 8. Resultados y alcance del procedimiento

Una vez concluidas ambas fases, se generó el archivo `Muestra_2_claveteada.xlsx`, que contiene las variables de la muestra homologada junto con los identificadores asignados.

La metodología final combinó dos estrategias complementarias: **una primera fase basada en similitudes ponderadas y una segunda fase de asignación mediante coincidencias de marca y tipo, priorizando el mayor valor comercial del catálogo**.

Las asignaciones iniciales se conservaron, mientras que el procedimiento complementario permitió evaluar nuevamente los registros que permanecieron sin clave. Posteriormente, se contabilizaron las asignaciones adicionales y se actualizó la cantidad total de vehículos identificados.

Debe señalarse que las claves asignadas representan correspondencias determinadas mediante los criterios establecidos en cada fase y no garantizan una identificación exacta de todas las características del vehículo.

De esta manera, el procedimiento permitió ampliar la cobertura del claveteo, manteniendo diferenciados los criterios de similitud global utilizados inicialmente y la regla complementaria de selección por valor comercial aplicada a los registros restantes.